In [2]:
import pickle

path = "/home/dacslab/lasse_jantsch/circuit_discovery/experiments/mib/results/eap_bilin_frnorm/ioi_gemma2_test_abs-False.pkl"

with open(path, "rb") as f:
    data = pickle.load(f)

data

{'area_under': 3.7301896881103516,
 'area_from_1': 2.7344216400146486,
 'average': 2.6479095458984374,
 'faithfulnesses': [0.374913330078125,
  0.47601806640625,
  0.9361328125,
  1.573203125,
  3.218125,
  4.6334375,
  4.717421875,
  4.7996875,
  4.75015625,
  1.0]}

In [1]:
import os
import sys

os.environ['CUDA_VISIBLE_DEVICES']='0'
proj_path = '/home/dacslab/lasse_jantsch/circuit_discovery'
if proj_path not in sys.path:
    sys.path.insert(0, proj_path)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm


from modular_transformer import patch_model_for_lvp
from modular_transformer.models import GPT2_ARC, LLAMA2_ARC, GEMMA2_ARC
from experiments.mib.data_utils import MIBDataset
from adapters import GPT2ModelAdapter, Gemma2ModelAdapter, Llama2ModelAdapter, ModelAdapter


MIB_MODEL_TO_HF_ID: dict[str, str] = {
    "gpt2": "gpt2",
    "qwen2.5": "Qwen/Qwen2.5-0.5B",
    "llama3": "meta-llama/Llama-3.1-8B",
    "gemma2": "google/gemma-2-2b",
}

MIB_MODEL_TO_ADAPTER_CLS: dict[str, type | None] = {
    "gpt2": GPT2ModelAdapter,
    "qwen2.5": Llama2ModelAdapter,
    "llama3": Llama2ModelAdapter,
    "gemma2": Gemma2ModelAdapter,
}

MIB_MODEL_TO_ARC: dict[str, type | None] = {
    "gpt2": GPT2_ARC,
    "qwen2.5": LLAMA2_ARC,
    "llama3": LLAMA2_ARC,
    "gemma2": GEMMA2_ARC,
}

CIRCUIT_DIR = '/home/dacslab/lasse_jantsch/circuit_discovery/experiments/mib/circuits'
PERCENTAGES = (.001, .002, .005, .01, .02, .05, .1, .2, .5, 1)

In [2]:
model_name = "qwen2.5"
model_id = MIB_MODEL_TO_HF_ID[model_name]

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"
if model_name == 'gpt2':
    tokenizer.padding_side = "right" # positional embeddings dont like left embedding
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if model_id != 'gpt2' else torch.float

model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=dtype, attn_implementation="eager", device_map="auto",
).eval()
model = patch_model_for_lvp(model, norm_approx='frozen') # lvp kwargs

adapter_cls: type[ModelAdapter] = MIB_MODEL_TO_ADAPTER_CLS[model_name]
adapter = adapter_cls(model, MIB_MODEL_TO_ARC[model_name], frozen_norm=False)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [3]:
task = "ioi"
dataset = MIBDataset(task, tokenizer, model_name, split='test', num_examples=200)
dataloader = dataset.dataloader(16)

In [4]:
def get_valid_edge_mask(adapter):
    mask = torch.zeros(adapter.source_dims, adapter.grad_dims).bool()
    mask[0] = True # -> emb
    for i in range(adapter.n_layers):
        attn_src = adapter.get_src_slice(type='attn', layer_id=i)
        attn_tgt = adapter.get_tgt_slice(type='attn_v', layer_id=i)
        mask[attn_src, attn_tgt.stop:] = True

        mlp_src = adapter.get_src_slice(type='mlp', layer_id=i)
        mlp_tgt = adapter.get_tgt_slice(type='mlp', layer_id=i)
        mask[mlp_src, mlp_tgt.stop:] = True
    
    return mask

def get_forward_to_backward(adapter):
    mask = torch.zeros(adapter.source_dims, adapter.grad_dims).bool()
    for i in range(adapter.n_layers):
        attn_src = adapter.get_src_slice(type='attn', layer_id=i)
        for qkv_type in ('attn_q', 'attn_k', 'attn_v'):
            attn_tgt = adapter.get_tgt_slice(type=qkv_type, layer_id=i)
            mask[attn_src, attn_tgt] = True

        mlp_src = adapter.get_src_slice(type='mlp', layer_id=i)
        mlp_tgt = adapter.get_tgt_slice(type='mlp', layer_id=i)
        mask[mlp_src, mlp_tgt] = True
    
    return mask


In [5]:
method = 'eap_bilin_frnorm'
absolute = False
circuit_path = os.path.join(CIRCUIT_DIR, method, f"{task}_{model_name}", 'scores.pt')
circuit_scores = torch.load(circuit_path, map_location='cpu')

if absolute:
    circuit_scores.abs_()

valid_edge_mask = get_valid_edge_mask(adapter)
n_edges = valid_edge_mask.sum().item()

forward_to_backward = get_forward_to_backward(adapter)

circuit_scores[~valid_edge_mask] = torch.finfo(circuit_scores.dtype).min
sorted_score_idx = torch.argsort(circuit_scores.view(-1), descending=True)

In [6]:
def prune(in_graph: torch.Tensor, forward_to_backward: torch.Tensor) -> torch.Tensor:
    nodes_in_graph = in_graph.any(dim=1)  # (n_forward,)

    changed = True
    while changed:
        nodes_with_outgoing = in_graph.any(dim=1)                                           # (n_forward,)
        nodes_with_ingoing = (in_graph.any(dim=0).float() @ forward_to_backward.float().T) > 0  # (n_forward,)
        nodes_with_ingoing[0] = True  # input node is always live

        new_nodes = nodes_with_outgoing & nodes_with_ingoing
        changed = not torch.equal(new_nodes, nodes_in_graph)
        nodes_in_graph = new_nodes

        backward_alive = (nodes_in_graph.float() @ forward_to_backward.float())  # (n_backward,)
        backward_alive[-1] = 1.0  # logits always live
        edge_mask = nodes_in_graph[:, None] & (backward_alive > 0)[None, :]      # (n_forward, n_backward)
        in_graph = in_graph & edge_mask

    return in_graph

def get_in_graph(circuit_scores, sorted_score_idx, forward_to_backward, percentage):
    k = int(n_edges * percentage)

    in_graph = torch.zeros_like(circuit_scores).bool()
    in_graph.view(-1)[sorted_score_idx[:k]] = True

    in_graph = prune(in_graph, forward_to_backward)

    # Flip boolean for all valid edges
    in_graph = ~in_graph
    in_graph[~valid_edge_mask] = False

    return in_graph

In [7]:
def update_source_cache(cache: torch.Tensor, tensor: torch.Tensor, adapter, type: str, layer_id: int = None):
    source_slice = adapter.get_src_slice(type=type, layer_id=layer_id)
    cache[source_slice] = tensor.detach()

def compute_attn_corrections(
    source: torch.Tensor,
    base_source: torch.Tensor,
    in_graph: torch.Tensor,
    adapter,
    layer_id: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute Q, K, V residual corrections in a single diff + einsum.
    Q/K/V share the same src_slice — diff is computed once, mask spans all 3*n_heads cols.
    Returns: (q_corr, k_corr, v_corr) each (n_heads, B, S, d)
    """
    src2d_slice = adapter.get_src_slice('attn_q', layer_id)
    src_slice = slice(None, src2d_slice.start)              # causal prefix
    q_tgt = adapter.get_tgt_slice('attn_q', layer_id)
    v_tgt = adapter.get_tgt_slice('attn_v', layer_id)
    qkv_tgt_slice = slice(q_tgt.start, v_tgt.stop)         # 3*n_heads consecutive cols
    diff = base_source[src_slice] - source[src_slice]       # (n_src, B, S, d)
    mask = in_graph[src_slice, qkv_tgt_slice].to(dtype=diff.dtype)  # (n_src, 3*n_heads)
    corr = torch.einsum('sBSd,st->tBSd', diff, mask)       # (3*n_heads, B, S, d)
    n = adapter.n_heads
    return corr[:n], corr[n:2*n], corr[2*n:]

def compute_patched_residual(
    source: torch.Tensor,
    base_source: torch.Tensor,
    in_graph: torch.Tensor,
    adapter,
    type: str,
    layer_id: int | None,
) -> torch.Tensor:
    """Compute residual correction for a single-target node (mlp or lm_head). Returns: (1, B, S, d)"""
    src2d_slice = adapter.get_src_slice(type, layer_id)
    tgt_slice = adapter.get_tgt_slice(type, layer_id)
    src_slice = slice(None, src2d_slice.start)              # causal prefix
    diff = base_source[src_slice] - source[src_slice]       # (n_src, B, S, d)
    mask = in_graph[src_slice, tgt_slice].to(dtype=diff.dtype)  # (n_src, 1)
    return torch.einsum('sBSd,st->tBSd', diff, mask)

# Logit difference at the last (non-padding) token position
def get_logit_diff(circuit_logits, clean_idx, corr_idx):
    B, S, _ = circuit_logits.shape
    last_token_idx = torch.full((B,), S - 1)
    logits_at_last = circuit_logits[range(B), last_token_idx]  # (B, vocab)
    return logits_at_last[range(B), clean_idx] - logits_at_last[range(B), corr_idx]

def get_scores(clean_logits, base_logits, patched_logits, clean_idx, corr_idx):
    clean_diff = get_logit_diff(clean_logits, clean_idx, corr_idx)
    base_diff = get_logit_diff(base_logits, clean_idx, corr_idx)
    patched_diff = get_logit_diff(patched_logits, clean_idx, corr_idx)
    return (patched_diff - base_diff) / (clean_diff - base_diff)

In [8]:
scores_agg = [[] for _ in PERCENTAGES]
for batch in tqdm(dataloader):
    clean_prompts, corrupt_prompts, clean_targets, corrupt_targets = batch

    clean_inputs = tokenizer(clean_prompts, padding=True, return_tensors='pt')
    corrupt_inputs = tokenizer(corrupt_prompts, padding=True, return_tensors='pt')

    curr_batch_size, curr_seq_len = clean_inputs['input_ids'].shape

    with model.session():

        base_source = torch.zeros(adapter.source_dims, curr_batch_size, curr_seq_len, adapter.model_dim,
                                  device=adapter.device, dtype=adapter.dtype)
        with model.trace(**corrupt_inputs):
            B, S, d = curr_batch_size, curr_seq_len, adapter.model_dim

            update_source_cache(
                base_source, adapter.embed_out_hook(detached=False).reshape(1, B, S, d), adapter, type='emb'
            )
            for layer_id, layer in enumerate(adapter.wrapped_layers):
                update_source_cache(
                    base_source, layer.head_wise_attn_out_hook(), adapter, type='attn', layer_id=layer_id)
                update_source_cache(
                    base_source, layer.mlp_out_hook().reshape(1, B, S, d), adapter, type='mlp', layer_id=layer_id)
            
            base_logits = model.output['logits']

        with model.trace(**clean_inputs):
            clean_logits = model.output['logits']

        for p_id, percentage in enumerate(PERCENTAGES[:-1]):

            in_graph = get_in_graph(circuit_scores, sorted_score_idx, forward_to_backward, percentage)
            in_graph = in_graph.to(device=adapter.device)

            source = torch.zeros(adapter.source_dims, curr_batch_size, curr_seq_len, adapter.model_dim,
                                 device=adapter.device, dtype=adapter.dtype).save()
            patch_cache = {}
            with model.trace(**clean_inputs):
                B, S, d = curr_batch_size, curr_seq_len, adapter.model_dim

                if adapter.uses_rotary_emb:
                    cos, sin = adapter.rotary_emb.output
                    patch_cache['rotary_emb'] = (cos.detach(), sin.detach())

                update_source_cache(
                    source, adapter.embed_out_hook(detached=False).reshape(1, B, S, d), adapter, type='emb'
                )
                for layer_id, layer in enumerate(adapter.wrapped_layers):
                    in_state = layer.residual_in_hook().unsqueeze(0)
                    q_corr, k_corr, v_corr = compute_attn_corrections(source, base_source, in_graph, adapter, layer_id)
                    layer.head_wise_query_patch_hook(in_state + q_corr, patch_cache)
                    layer.head_wise_key_patch_hook(in_state + k_corr, patch_cache)
                    layer.head_wise_value_patch_hook(in_state + v_corr, patch_cache)
                    update_source_cache(
                        source, layer.head_wise_attn_out_hook(), adapter, type='attn', layer_id=layer_id)

                    mid_state = layer.residual_mid_hook()
                    layer.mlp_patch_hook(
                        mid_state + compute_patched_residual(source, base_source, in_graph, adapter, 'mlp', layer_id)[0],
                    )
                    update_source_cache(
                        source, layer.mlp_out_hook().reshape(1, B, S, d), adapter, type='mlp', layer_id=layer_id)
                    out_state = layer.residual_out_hook()
                    
                adapter.logit_patch_hook(
                    out_state + compute_patched_residual(source, base_source, in_graph, adapter, 'lm_head', None)[0],
                )

                patched_logits = model.output['logits']

                scores = get_scores(clean_logits, base_logits, patched_logits, clean_targets, corrupt_targets)
                scores_agg[p_id].extend(scores.tolist())

        scores_agg[-1].extend([1 for _ in range(B)])

100%|██████████| 13/13 [00:16<00:00,  1.29s/it]


In [9]:
torch.tensor(scores_agg[:-1]).mean(-1)

tensor([0.5756, 0.7567, 1.1359, 1.5637, 1.7355, 1.9240, 2.0220, 2.0621, 2.1033])

In [10]:
import math

faithfulnesses = [sum(s) / len(s) for s in scores_agg]

area_under = 0.
area_from_1 = 0.
for i in range(len(faithfulnesses) - 1):
    x1, x2 = PERCENTAGES[i], PERCENTAGES[i + 1]
    y1, y2 = faithfulnesses[i], faithfulnesses[i + 1]
    w = x2 - x1
    area_under  += w * (y1 + y2) / 2
    area_from_1 += w * (abs(1. - y1) + abs(1. - y2)) / 2

print(f"CPR (area_under):  {area_under:.4f}")
print(f"CMD (area_from_1): {area_from_1:.4f}")
print(f"Average:           {sum(faithfulnesses) / len(faithfulnesses):.4f}")

CPR (area_under):  1.7851
CMD (area_from_1): 0.7875
Average:           1.4879


In [11]:
log_area_under = 0.
log_area_from_1 = 0.
for i in range(len(faithfulnesses) - 1):
    x1, x2 = math.log(PERCENTAGES[i]), math.log(PERCENTAGES[i + 1])
    y1, y2 = faithfulnesses[i], faithfulnesses[i + 1]
    w = x2 - x1
    log_area_under  += w * (y1 + y2) / 2
    log_area_from_1 += w * (abs(1. - y1) + abs(1. - y2)) / 2

print(f"CPR log (area_under):  {log_area_under:.4f}")
print(f"CMD log (area_from_1): {log_area_from_1:.4f}")

CPR log (area_under):  10.8514
CMD log (area_from_1): 4.6294
